# TRAINING

In [ ]:
import zipfile
import os

# Extract train metadata
with zipfile.ZipFile('train_allmetadata_json.zip', 'r') as zip_ref:
    zip_ref.extractall('train_metadata')

# Extract test metadata
with zipfile.ZipFile('test_allmetadata_json.zip', 'r') as zip_ref:
    zip_ref.extractall('test_metadata')

print("Extraction done!")

Extraction done!


In [ ]:
print(os.listdir('train_metadata')[:5])
print(os.listdir('test_metadata')[:5])

['train_allmetadata_json', '__MACOSX']
['test_allmetadata_json', '__MACOSX']


In [ ]:
import pandas as pd
import json
import glob

json_files = glob.glob('train_metadata/**/*.json', recursive=True)
data_frames = []

for file in json_files:
    with open(file, 'r') as f:
        data = json.load(f)
        # dictionary to DataFrame
        temp_df = pd.DataFrame.from_dict(data)
        # Ensure a consistent integer-based RangeIndex for proper concatenation
        temp_df = temp_df.reset_index(drop=True)
        data_frames.append(temp_df)

if data_frames:
    final_df = pd.concat(data_frames, axis=1)

    # Duplicate columns removal
    final_df = final_df.loc[:,~final_df.columns.duplicated()]
else:
    print("No JSON files found!")

with open('train_label.txt', 'r') as f:
    labels = [float(x.strip()) for x in f.readlines()]

# Adding Labels to dataframe
final_df['label'] = labels[:len(final_df)]

print("Final Shape:", final_df.shape)
print(final_df.head())

Final Shape: (305613, 24)
        Uid   Pid photo_firstdate  photo_count  ispro timezone_offset  \
0    59@N75   775            None       1429.0    1.0            None   
1     1@N18  1075            None      43108.0    0.0          -03:00   
2   351@N64  4890            None       1035.0    1.0          +01:00   
3     6@N59  6568            None      83322.0    1.0          +01:00   
4  1617@N40  7079            None       5958.0    1.0          -02:00   

  photo_firstdatetaken                                       timezone_id  \
0  1994-01-01 19:36:55                                              None   
1  1980-01-01 00:00:23                                          Brasilia   
2  1950-01-07 22:55:16  Amsterdam, Berlin, Bern, Rome, Stockholm, Vienna   
3  2001-01-05 01:56:02  Amsterdam, Berlin, Bern, Rome, Stockholm, Vienna   
4  1998-11-25 19:11:38                                      Mid-Atlantic   

                                    user_description     location_description 

In [ ]:
print(f"Total number of columns: {len(final_df.columns)}")
print(f"Shape of the DataFrame: {final_df.shape}")

Total number of columns: 24
Shape of the DataFrame: (305613, 24)


# Training Statistics

In [ ]:
# 1. Total Posts
total_posts = len(final_df)

# 2. Total Unique Users
total_users = final_df['Uid'].nunique()

# 3. Total Categories
total_categories = final_df['Category'].nunique()

# 4. Total Customized Tags
all_tags_list = final_df['Alltags'].str.split().explode()
total_unique_tags = all_tags_list.nunique()

# 5. Average Title Length (Avg. Title Length)
avg_title_length = final_df['Title'].str.len().mean()

# Summary Table
stats_summary = pd.DataFrame({
    'Dataset': ['SMPD'],
    '#Post': [total_posts],
    '#User': [total_users],
    '#Categories': [total_categories],
    'Avg. Title Length': [round(avg_title_length, 2)],
    '#Customize Tags': [total_unique_tags]
})

print("--- Dataset Statistics ---")
print(stats_summary.to_string(index=False))

--- Dataset Statistics ---
Dataset  #Post  #User  #Categories  Avg. Title Length  #Customize Tags
   SMPD 305613  38312           11              27.68           234992


# TESTING

In [ ]:
import pandas as pd
import json
import glob

# 1. Test Metadata folder ko scan karein
test_files = glob.glob('test_metadata/**/*.json', recursive=True)
test_data_frames = []

print(f"Found {len(test_files)} JSON files in test metadata.")

for file in test_files:
    with open(file, 'r') as f:
        try:
            data = json.load(f)
            # dictionary to DataFrame
            temp_df = pd.DataFrame.from_dict(data)
            # Reset index to ensure alignment during concat
            temp_df = temp_df.reset_index(drop=True)
            test_data_frames.append(temp_df)
        except Exception as e:
            print(f"Error loading {file}: {e}")

if test_data_frames:
    test_df = pd.concat(test_data_frames, axis=1)

    # Duplicate columns (Uid, Pid etc.) removal
    test_df = test_df.loc[:, ~test_df.columns.duplicated()]

    print("Test Data extraction successful.")
else:
    print("No JSON files found in test_metadata!")

# 2. Test Labels
try:
    with open('test_label.txt', 'r') as f:
        test_labels = [float(x.strip()) for x in f.readlines()]
    test_df['label'] = test_labels[:len(test_df)]
    print("Test labels attached.")
except FileNotFoundError:
    print("Note: test_label.txt not found. Test set will have no label column.")

# 3. Final Check
print("\n--- Test Set Summary ---")
print("Final Shape:", test_df.shape)
print(test_df.head())

print("\nUnique Users in Test:", test_df['Uid'].nunique() if 'Uid' in test_df else "N/A")

Found 5 JSON files in test metadata.
Test Data extraction successful.
Note: test_label.txt not found. Test set will have no label column.

--- Test Set Summary ---
Final Shape: (180581, 23)
        Category   Concept     Pid        Uid Subcategory  \
0  Social&People      hugs   50731   8770@N36        Love   
1  Entertainment   imagine   96454  10082@N16       Books   
2           Food    hungry  172136  10082@N16     Dessert   
3  Entertainment  bookworm  177707  24607@N38       Books   
4  Entertainment  bookworm  177708  24607@N38       Books   

                                             Alltags Mediatype  \
0  vintage toys cool nikon peace lego free retro ...     photo   
1  sky people musician paris tower john poetry ki...     photo   
2  paris seine river skeleton fossil rivedroite d...     photo   
3  reading book community library libraries maine...     photo   
4  reading book community library libraries maine...     photo   

                                              

# Testing Statistics

In [ ]:
import pandas as pd
import json
import glob

test_files = glob.glob('test_metadata/**/*.json', recursive=True)
test_data_frames = []

for file in test_files:
    with open(file, 'r') as f:
        try:
            data = json.load(f)
            temp_df = pd.DataFrame.from_dict(data)
            temp_df = temp_df.reset_index(drop=True)
            test_data_frames.append(temp_df)
        except Exception as e:
            print(f"Error loading {file}: {e}")

if test_data_frames:
    test_df = pd.concat(test_data_frames, axis=1)
    test_df = test_df.loc[:, ~test_df.columns.duplicated()]

    # --- Statistics Calculation ---
    print("\n--- Testing Dataset Statistics ---")

    #Post
    test_posts = len(test_df)

    #User
    test_users = test_df['Uid'].nunique() if 'Uid' in test_df else 0

    #Categories
    test_categories = test_df['Subcategory'].nunique() if 'Subcategory' in test_df else 0

    # Avg. Title Length
    test_avg_title_len = test_df['Title'].str.len().mean() if 'Title' in test_df else 0

    # #Customize Tags
    if 'Alltags' in test_df:
        test_all_tags = test_df['Alltags'].astype(str).str.split().explode()
        test_unique_tags = test_all_tags.nunique()
    else:
        test_unique_tags = 0

    # Summary Table
    test_stats_summary = pd.DataFrame({
        'Dataset': ['SMPD_Test'],
        '#Post': [test_posts],
        '#User': [test_users],
        '#Categories': [test_categories],
        'Avg. Title Length': [round(test_avg_title_len, 2)],
        '#Customize Tags': [test_unique_tags]
    })

    print(test_stats_summary.to_string(index=False))
    print("\nTest DataFrame Shape:", test_df.shape)

else:
    print("No test files found! Path check karein.")


--- Testing Dataset Statistics ---
  Dataset  #Post  #User  #Categories  Avg. Title Length  #Customize Tags
SMPD_Test 180581  31392           77              27.25           156622

Test DataFrame Shape: (180581, 23)


# COMBINED STATS OF TRAINING AND TESTING

In [ ]:
import pandas as pd
import numpy as np

full_df = pd.concat([final_df, test_df], axis=0).reset_index(drop=True)

print(f"Combined Shape: {full_df.shape}")

# --- Statistics Calculation ---

#Post
total_posts = len(full_df)

# #User
total_users = full_df['Uid'].nunique()

# #Categories
total_categories = full_df['Concept'].nunique()

# Avg. Title Length
avg_title_len = full_df['Title'].astype(str).str.len().mean()

# Unique Tags calculation
all_tags = full_df['Alltags'].astype(str).str.split().explode()
total_unique_tags = all_tags.nunique()

# Temporal Range (Months)
postdate_numeric = pd.to_numeric(full_df['Postdate'], errors='coerce')
full_df['Postdate_dt'] = pd.to_datetime(postdate_numeric, unit='s')

min_date = full_df['Postdate_dt'].min()
max_date = full_df['Postdate_dt'].max()

# Months difference ka formula
months_range = (max_date.year - min_date.year) * 12 + (max_date.month - min_date.month)

total_stats = pd.DataFrame({
    'Dataset': ['SMPD (Total)'],
    '#Post': [f"{total_posts/1000:.0f}k" if total_posts > 400000 else total_posts],
    '#User': [total_users],
    '#Categories': [total_categories],
    'Temporal Range': [f"{months_range} Months"],
    'Avg. Title Length': [int(round(avg_title_len, 0))],
    '#Customize Tags': [total_unique_tags]
})

print("\n--- Final Combined Statistics (Matching Research Paper) ---")
print(total_stats.to_string(index=False))


Combined Shape: (486194, 24)

--- Final Combined Statistics (Matching Research Paper) ---
     Dataset #Post  #User  #Categories Temporal Range  Avg. Title Length  #Customize Tags
SMPD (Total)  486k  60093          669      16 Months                 28           324856


# Transforming social media metadata into semantically enriched and contextual text representations

In [ ]:
import pandas as pd

def apply_semantic_enrichment(row):
    r = row.fillna("Not Available")

    # --- STEP 1: Define Field Descriptions (As per Table 3) ---
    desc_uid = f"the user this post belongs to is {r['Uid']}"
    desc_pid = f"the photo along with the post is {r['Pid']}"
    desc_cat = f"the category of the post is {r['Category']}"
    desc_sub = f"the subcategory of the post is {r['Subcategory']}"
    desc_con = f"the concept of the post is {r['Concept']}"
    desc_title = f"the title of the post defined by the user is '{r['Title']}'"
    desc_tags = f"the customized tags from users are {r['Alltags']}"
    desc_media = f"the type of the attached media file is {r['Mediatype']}"
    desc_date = f"the publish timestamp of the post is {r['Postdate_dt']}"
    desc_count = f"the number of posted photos by the user is {r['photo_count']}"
    desc_pro = f"the user belongs to pro member: {r['ispro']}"

    # --- STEP 2: Instruction Component ---
    instruction = (
        "Instruction: You are a social media analyst. Analyze the provided metadata "
        "and predict the popularity score (log-scale view count) for this post. "
        "The output should be a single numerical value."
    )

    # --- STEP 3: Template 1 (Systematic Key-Value Structure) ---
    template_1_input = (
        f"Input: [User Info: {desc_uid}, {desc_pro}, {desc_count}]. "
        f"[Post Info: {desc_pid}, {desc_title}, {desc_cat}, {desc_sub}, {desc_con}, "
        f"{desc_tags}, {desc_media}, {desc_date}]."
    )

    # --- STEP 4: Template 2 (Liberal/Narrative Approach) ---
    template_2_input = (
        f"Input: A {r['ispro']} pro user (ID: {r['Uid']}) who has uploaded {r['photo_count']} photos "
        f"posted a {r['Mediatype']} with the title '{r['Title']}'. This post belongs to the "
        f"{r['Category']} category, specifically focusing on {r['Concept']}. It was tagged with "
        f"{r['Alltags']} and published at {r['Postdate_dt']}. Analyze this context to predict popularity."
    )

    return pd.Series([
        f"{instruction}\n{template_1_input}",
        f"{instruction}\n{template_2_input}"
    ])

# Apply transformation to the entire dataframe
print("Applying Semantic Enrichment (Templates 1 & 2)...")
full_df[['Final_Prompt_T1', 'Final_Prompt_T2']] = full_df.apply(apply_semantic_enrichment, axis=1)

# Check results
print("\n--- [REPLICATED] Template 1: Systematic ---")
print(full_df['Final_Prompt_T1'].iloc[0])

print("\n--- [REPLICATED] Template 2: Narrative ---")
print(full_df['Final_Prompt_T2'].iloc[0])

print("Saving enriched data to disk...")

save_cols = ['Final_Prompt_T1', 'Final_Prompt_T2', 'label']
full_df[save_cols].to_parquet('enriched_smpd_data.parquet', index=False)

print("File saved as 'enriched_smpd_data.parquet'!")



Applying Semantic Enrichment (Templates 1 & 2)...

--- [REPLICATED] Template 1: Systematic ---
Instruction: You are a social media analyst. Analyze the provided metadata and predict the popularity score (log-scale view count) for this post. The output should be a single numerical value.
Input: [User Info: the user this post belongs to is 59@N75, the user belongs to pro member: 1.0, the number of posted photos by the user is 1429.0]. [Post Info: the photo along with the post is 775, the title of the post defined by the user is 'Luis Drayton - Edinburgh shoot #6', the category of the post is Fashion, the subcategory of the post is Fashion, the concept of the post is glam, the customized tags from users are rock punk transgender tranny electronicmusic electro glam electronica luisdrayton fusionrecords thefusionnetwork lmwcphotography, the type of the attached media file is photo, the publish timestamp of the post is 2015-10-28 07:19:38].

--- [REPLICATED] Template 2: Narrative ---
Instruc

In [ ]:
import pandas as pd

full_df = pd.read_parquet('enriched_smpd_data.parquet')

# Lightweight training config 
MAX_TRAIN_SAMPLES = 50000
MAX_EVAL_SAMPLES = 5000   # Keeping eval smaller for faster validation cycles
RANDOM_SEED = 42

# Shuffle once so first rows are not biased by source ordering
full_df = full_df.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)

# Keep only rows with valid prompt + label
full_df = full_df.dropna(subset=['Final_Prompt_T1', 'label']).reset_index(drop=True)

train_end = min(MAX_TRAIN_SAMPLES, len(full_df))
eval_end = min(train_end + MAX_EVAL_SAMPLES, len(full_df))

train_df_small = full_df.iloc[:train_end].copy()
eval_df_small = full_df.iloc[train_end:eval_end].copy()

# Fallback split if dataset is smaller than requested
if len(eval_df_small) == 0 and len(train_df_small) > 1:
    split_idx = max(1, int(0.8 * len(train_df_small)))
    eval_df_small = train_df_small.iloc[split_idx:].copy()
    train_df_small = train_df_small.iloc[:split_idx].copy()

if len(train_df_small) == 0 or len(eval_df_small) == 0:
    raise ValueError(
        'Not enough valid rows after filtering. Check parquet content and required columns.'
    )

print('Data loaded and reduced for lighter fine-tuning.')
print(f'Full rows: {len(full_df)}')
print(f'Train rows used: {len(train_df_small)}')
print(f'Eval rows used: {len(eval_df_small)}')

Data loaded and reduced for lighter fine-tuning.
Full rows: 305613
Train rows used: 50000
Eval rows used: 5000


# LoRA and Model Setup

In [ ]:
# ============================================================
# CELL 1 — Install correct versions (run first, then restart)
# ============================================================
#!pip install -q \
 #  "transformers==4.44.0" \
  #  "peft==0.11.1" \
   # "bitsandbytes==0.43.1" \
   # "accelerate==0.31.0" \
   # "datasets==2.20.0" \
    #"trl==0.9.4"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 115.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.4/309.4 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.7/226.7 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.1/316.1 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# ============================================================
# Imports
# ============================================================
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

In [ ]:
# ============================================================
# Data Preparation
# ============================================================
def prepare_dataset(df):
    return Dataset.from_dict({
        "text": [
            f"{prompt} Output: {label}"
            for prompt, label in zip(df["Final_Prompt_T1"], df["label"])
        ]
    })

train_data = prepare_dataset(train_df_small)
eval_data = prepare_dataset(eval_df_small)

print(f"Train: {len(train_data)} | Eval: {len(eval_data)}")

Train: 50000 | Eval: 5000


In [ ]:
# ============================================================
# Tokenizer
# ============================================================
USE_LIGHT_MODEL = True  # Set True to use 1.5B model for much faster/lighter training
model_id = "Qwen/Qwen2-1.5B-Instruct" if USE_LIGHT_MODEL else "Qwen/Qwen2-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"   # Required for causal LM training

print(f"Using base model: {model_id}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Using base model: Qwen/Qwen2-1.5B-Instruct


In [ ]:
# ============================================================
# 4-bit Quantisation Config
# ============================================================
compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
    else torch.float16
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

In [3]:
 !pip install -q "bitsandbytes>=0.46.1" "transformers==4.44.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 82.7 MB/s eta 0:00:00


In [ ]:
!pip install -q "accelerate==0.34.0" "bitsandbytes>=0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 28.0 MB/s eta 0:00:00


In [ ]:
# ============================================================
# Load Model 
# ============================================================
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
 )

model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

print("Model loaded successfully!")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

drive_save_path = "/content/drive/MyDrive/saved-model"

if not os.path.exists(drive_save_path):
    os.makedirs(drive_save_path)

In [ ]:
model.save_pretrained(drive_save_path)
tokenizer.save_pretrained(drive_save_path)

print(f"✅ Model successfully saved to Drive at: {drive_save_path}")

✅ Model successfully saved to Drive at: /content/drive/MyDrive/saved-model


In [8]:
save_path = "./saved_model"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("Model saved successfully!")

Model saved successfully!


In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch


model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto"
 )

model = PeftModel.from_pretrained(model, load_path)

print("✅ Model successfully loaded from Drive!")

In [ ]:
# ============================================================
# LoRA Config
# ============================================================
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"], 
    lora_dropout=0.0,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705


In [ ]:
# ============================================================
# Tokenise Dataset
# ============================================================
MAX_LENGTH = 192  # Lighter than 256; 

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

tokenized_train = train_data.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)
tokenized_eval = eval_data.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
save_path = "./qwen2-popularity-model"

In [13]:
model.enable_input_require_grads()
model.train()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2SdpaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear4bit(in_features

In [ ]:
# ============================================================
# Training Arguments (Balanced lightweight setup)
# ============================================================
use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/qwen2-output",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    max_steps=800,                
    logging_steps=20,
    evaluation_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    bf16=use_bf16,
    fp16=not use_bf16,
    optim="paged_adamw_8bit",
    report_to="none",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    group_by_length=True,
)

/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
# ============================================================
# Train & Save
# ============================================================
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
)

print("Starting Fine-Tuning...")
trainer.train()

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Done! Model saved to {save_path}")

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
max_steps is given, it will override any value given in num_train_epochs


Starting Fine-Tuning...


Step,Training Loss,Validation Loss
200,1.127300,1.102430
400,1.101700,1.068514
600,1.057000,1.056886
800,1.065500,1.054675


Done! Model saved to ./qwen2-popularity-model


In [ ]:
import re
import torch
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Re-define apply_semantic_enrichment function for this cell
def apply_semantic_enrichment(row):
    r = row.fillna("Not Available")

    # --- STEP 1: Define Field Descriptions (As per Table 3) ---
    desc_uid = f"the user this post belongs to is {r['Uid']}"
    desc_pid = f"the photo along with the post is {r['Pid']}"
    desc_cat = f"the category of the post is {r['Category']}"
    desc_sub = f"the subcategory of the post is {r['Subcategory']}"
    desc_con = f"the concept of the post is {r['Concept']}"
    desc_title = f"the title of the post defined by the user is '{r['Title']}'"
    desc_tags = f"the customized tags from users are {r['Alltags']}"
    desc_media = f"the type of the attached media file is {r['Mediatype']}"
    desc_date = f"the publish timestamp of the post is {r['Postdate_dt']}"
    desc_count = f"the number of posted photos by the user is {r['photo_count']}"
    desc_pro = f"the user belongs to pro member: {r['ispro']}"

    # --- STEP 2: Instruction Component ---
    instruction = (
        '''Instruction:
          You are an expert social media analyst specializing in engagement prediction.

          Task:
          Given the metadata of a post, predict its popularity score as the log10 of expected view count.

          Guidelines:
          - Carefully analyze ALL metadata fields (e.g., title, hashtags, description, timing, platform signals).
          - Identify patterns that increase or decrease engagement (e.g., trending topics, emotional tone, clickbait, hashtags, posting time).
          - Avoid default or average predictions — each prediction MUST reflect the uniqueness of the input.
          - Use a wide range of values depending on the content quality and virality potential.
          - Ensure predictions vary meaningfully across different inputs.

          Output Format:
          - Return ONLY a single floating-point number (e.g., 3.72)
          - Do NOT include explanations or text.

          Now analyze the following metadata and predict the score:'''
    )

    # --- STEP 3: Template 1 (Systematic Key-Value Structure) ---
    template_1_input = (
        f"Input: [User Info: {desc_uid}, {desc_pro}, {desc_count}]. "
        f"[Post Info: {desc_pid}, {desc_title}, {desc_cat}, {desc_sub}, {desc_con}, "
        f"{desc_tags}, {desc_media}, {desc_date}]."
    )

    # --- STEP 4: Template 2 (Liberal/Narrative Approach) ---
    template_2_input = (
        f"Input: A {r['ispro']} pro user (ID: {r['Uid']}) who has uploaded {r['photo_count']} photos "
        f"posted a {r['Mediatype']} with the title '{r['Title']}'. This post belongs to the "
        f"{r['Category']} category, specifically focusing on {r['Concept']}. It was tagged with "
        f"{r['Alltags']} and published at {r['Postdate_dt']}. Analyze this context to predict popularity."
    )

    return pd.Series([
        f"{instruction}\n{template_1_input}",
        f"{instruction}\n{template_2_input}"
    ])

# ============================================================
# Prediction Function
# ============================================================
def predict_popularity(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,        
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Number extraction
    numbers = re.findall(r'[-+]?\d*\.?\d+', response)
    if numbers:
        try:
            val = float(numbers[0])
            if 0 <= val <= 30:    # Reasonable range
                return val
        except:
            pass
    return 5.0    # Default fallback

# ============================================================
# Eval Dataset — 200 proper samples
# ============================================================

sampled_df = final_df.sample(n=2400, random_state=42) # Using a random_state for reproducibility

eval_pool = final_df[~final_df.index.isin(sampled_df.index)]
eval_samples = eval_pool.sample(200, random_state=99).reset_index(drop=True)


pd_num = pd.to_numeric(eval_samples['Postdate'], errors='coerce')
eval_samples['Postdate_dt'] = pd.to_datetime(pd_num, unit='s', errors='coerce')

eval_samples[['Final_Prompt_T1', 'Final_Prompt_T2']] = eval_samples.apply(
    apply_semantic_enrichment, axis=1
)

print(f"Eval samples ready: {len(eval_samples)}")
print("Predictions start: \n")

# ============================================================
# Predictions
# ============================================================
actual_scores    = []
predicted_scores = []

for i, row in eval_samples.iterrows():
    pred = predict_popularity(row['Final_Prompt_T2'])
    actual_scores.append(row['label'])
    predicted_scores.append(pred)

    idx = len(actual_scores)
    if idx % 25 == 0:
        print(f"  {idx}/200 done | Actual: {row['label']:.2f} | Pred: {pred:.2f}")

# ============================================================
# Final Metrics
# ============================================================
spr, _ = spearmanr(actual_scores, predicted_scores)
mae    = mean_absolute_error(actual_scores, predicted_scores)
mse    = mean_squared_error(actual_scores, predicted_scores)

print("\n" + "="*50)
print("       FINAL EVALUATION RESULTS")
print("="*50)
print(f"  SPR (Spearman): {spr:.4f}  (paper: 0.8866)")
print(f"  MAE           : {mae:.4f}  (paper: 0.6432)")
print(f"  MSE           : {mse:.4f}  (paper: 1.4223)")
print("="*50)

Eval samples ready: 200
Predictions start: 



/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


  25/200 done | Actual: 3.32 | Pred: 5.00
  50/200 done | Actual: 13.30 | Pred: 5.00
  75/200 done | Actual: 4.64 | Pred: 5.00
  100/200 done | Actual: 5.39 | Pred: 5.00
  125/200 done | Actual: 7.30 | Pred: 5.00
  150/200 done | Actual: 4.00 | Pred: 5.00
  175/200 done | Actual: 1.58 | Pred: 5.00
  200/200 done | Actual: 8.22 | Pred: 5.00

       FINAL EVALUATION RESULTS
  SPR (Spearman): -0.0357  (paper: 0.8866)
  MAE           : 2.2178  (paper: 0.6432)
  MSE           : 8.0837  (paper: 1.4223)
